# Hyperparameter Sweep Heatmaps
Visualize metrics across a 2D hyperparameter grid for a chosen layer.

In [ ]:
# ============== CONFIG ==============
SWEEP_DIR = "../logdir/ego_sweep3"                         # folder containing sweep run subdirs
RUN_PREFIX = "ego_sweep3"                        # only use subdirs starting with this
PARAM_PATTERN = "l1wm{l1wm}_l1pol{l1pol}_l1valWM{l1val}"  # {name} placeholders for all hyperparams
GRID_AXES = ("l1wm", "l1pol")                    # which 2 params become heatmap axes
FILTER = {"l1val": "l1wm"}                       # keep runs where l1val == l1wm
                                                 # change to {"l1val": "l1pol"} for second condition
LAYER = "dyn/deter"                              # layer to plot

DECODE_SUBDIR = "layer_decoder_results"          # subfolder with layer_decode_results.pkl
TUNING_SUBDIR = "tuning_results"                 # subfolder with tuning_results.pkl

EV_PERCENTILE = 80                               # top-EV percentile for filtered SI
# ====================================

In [ ]:
import os, re, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning)

# --- Parse param pattern into regex ---
def build_regex(prefix, pattern):
    """Convert e.g. 'l1wm{l1wm}_l1pol{l1pol}' into a named-group regex."""
    parts = re.split(r'\{(\w+)\}', pattern)
    regex = re.escape(prefix) + r'_'
    for i, part in enumerate(parts):
        if i % 2 == 0:
            regex += re.escape(part)
        else:
            regex += rf'(?P<{part}>[^_/]+)'
    return re.compile(regex + r'$')

def parse_float(s):
    """Parse strings like '1e-2', '1e0' to float."""
    return float(s)

# --- Discover runs ---
sweep_path = Path(SWEEP_DIR)
pattern_re = build_regex(RUN_PREFIX, PARAM_PATTERN)
all_param_names = re.findall(r'\{(\w+)\}', PARAM_PATTERN)
assert len(GRID_AXES) == 2, f"GRID_AXES must have exactly 2 entries, got {GRID_AXES}"
assert all(ax in all_param_names for ax in GRID_AXES), \
    f"GRID_AXES {GRID_AXES} must be a subset of pattern params {all_param_names}"
param_names = list(GRID_AXES)

runs = {}  # (p1_val, p2_val) -> path
skipped = []
for d in sorted(sweep_path.iterdir()):
    if not d.is_dir():
        continue
    m = pattern_re.match(d.name)
    if not m:
        continue
    # Apply filter: e.g. {"l1val": "l1wm"} keeps runs where l1val == l1wm
    parsed = {name: parse_float(m.group(name)) for name in all_param_names}
    skip = False
    for filt_param, filt_ref in FILTER.items():
        if filt_ref in parsed:
            # Compare to another param's value
            if parsed[filt_param] != parsed[filt_ref]:
                skip = True
                break
        else:
            # Compare to a literal value
            if parsed[filt_param] != parse_float(filt_ref):
                skip = True
                break
    if skip:
        skipped.append(d.name)
        continue
    key = (parsed[param_names[0]], parsed[param_names[1]])
    runs[key] = d

p1_vals = sorted(set(k[0] for k in runs))
p2_vals = sorted(set(k[1] for k in runs))
filter_desc = ", ".join(f"{k}=={v}" for k, v in FILTER.items())
print(f"Filter: {filter_desc} → kept {len(runs)}, skipped {len(skipped)}")
print(f"Found {len(runs)} runs: {param_names[0]}={p1_vals}, {param_names[1]}={p2_vals}")
print(f"Grid: {len(p1_vals)} x {len(p2_vals)}")

In [3]:
# --- Load data ---
decode_data = {}  # (p1, p2) -> layer_fold_values array
tuning_data = {}  # (p1, p2) -> layer dict from tuning_results

for key, run_path in runs.items():
    # Decode results
    decode_pkl = run_path / DECODE_SUBDIR / 'layer_decode_results.pkl'
    if decode_pkl.exists():
        with open(decode_pkl, 'rb') as f:
            d = pickle.load(f)
        if LAYER in d.get('layer_fold_values', {}):
            decode_data[key] = d['layer_fold_values'][LAYER]

    # Tuning results
    tuning_pkl = run_path / TUNING_SUBDIR / 'tuning_results.pkl'
    if tuning_pkl.exists():
        with open(tuning_pkl, 'rb') as f:
            t = pickle.load(f)
        if LAYER in t.get('layers', {}):
            tuning_data[key] = t['layers'][LAYER]

print(f"Decode loaded: {len(decode_data)}/{len(runs)}, Tuning loaded: {len(tuning_data)}/{len(runs)}")
missing_decode = set(runs) - set(decode_data)
missing_tuning = set(runs) - set(tuning_data)
if missing_decode:
    print(f"Missing decode: {[runs[k].name for k in missing_decode]}")
if missing_tuning:
    print(f"Missing tuning: {[runs[k].name for k in missing_tuning]}")

Decode loaded: 16/16, Tuning loaded: 16/16


In [4]:
# --- Load training performance (scores.jsonl) ---
import json

SMOOTH_WINDOW = 50  # rolling mean window for episode score (matches plot_training.py)

score_data = {}  # (p1, p2) -> dict with 'mean_score', 'crafter_score'

for key, run_path in runs.items():
    scores_file = run_path / 'scores.jsonl'
    if not scores_file.exists():
        continue
    episodes = []
    with open(scores_file) as f:
        for line in f:
            try:
                episodes.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    if not episodes:
        continue

    # Smoothed episode score: rolling mean, take final value
    all_scores = np.array([ep['episode/score'] for ep in episodes if 'episode/score' in ep])
    if len(all_scores) >= SMOOTH_WINDOW:
        kernel = np.ones(SMOOTH_WINDOW) / SMOOTH_WINDOW
        smoothed = np.convolve(all_scores, kernel, mode='valid')
        mean_score = smoothed[-1]
    elif len(all_scores) > 0:
        mean_score = np.mean(all_scores)
    else:
        mean_score = None

    # Crafter score = geometric mean of per-achievement success rates (last SMOOTH_WINDOW episodes)
    last_eps = episodes[-SMOOTH_WINDOW:]
    achievement_keys = [k for k in last_eps[0] if k.startswith('episode/achievement_')]
    if achievement_keys:
        success_rates = []
        for ak in achievement_keys:
            vals = [ep[ak] for ep in last_eps if ak in ep]
            rate = np.mean(vals) if vals else 0.0
            success_rates.append(rate + 1e-6)  # avoid log(0)
        crafter_score = np.exp(np.mean(np.log(success_rates))) * 100  # percentage
    else:
        crafter_score = None

    score_data[key] = {'mean_score': mean_score, 'crafter_score': crafter_score}

print(f"Scores loaded: {len(score_data)}/{len(runs)}")
missing_scores = set(runs) - set(score_data)
if missing_scores:
    print(f"Missing scores: {[runs[k].name for k in missing_scores]}")

Scores loaded: 16/16


In [5]:
# --- Helpers (grid building + plotting) ---
def make_grid(p1_vals, p2_vals, data_dict, extract_fn):
    """Build a 2D numpy array (len(p1) x len(p2)) from a dict keyed by (p1,p2)."""
    grid = np.full((len(p1_vals), len(p2_vals)), np.nan)
    for (i, p1) in enumerate(p1_vals):
        for (j, p2) in enumerate(p2_vals):
            key = (p1, p2)
            if key in data_dict:
                val = extract_fn(data_dict[key])
                if val is not None:
                    grid[i, j] = val
    return grid

def format_tick(v):
    """Format float as compact scientific notation."""
    if v == 0:
        return '0'
    exp = int(np.floor(np.log10(abs(v))))
    coeff = v / 10**exp
    if abs(coeff - 1.0) < 1e-6:
        return f'1e{exp}'
    return f'{coeff:.0f}e{exp}'

p1_labels = [format_tick(v) for v in p1_vals]
p2_labels = [format_tick(v) for v in p2_vals]

def plot_heatmap(ax, grid, title, cmap, fmt='.3f'):
    im = ax.imshow(grid, cmap=cmap, aspect='equal', origin='lower')
    ax.set_xticks(range(len(p2_vals)))
    ax.set_xticklabels(p2_labels)
    ax.set_yticks(range(len(p1_vals)))
    ax.set_yticklabels(p1_labels)
    ax.set_xlabel(param_names[1])
    ax.set_ylabel(param_names[0])
    ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            val = grid[i, j]
            if np.isfinite(val):
                norm_val = (val - np.nanmin(grid)) / (np.nanmax(grid) - np.nanmin(grid) + 1e-12)
                color = 'white' if norm_val > 0.6 or norm_val < 0.15 else 'black'
                ax.text(j, i, f'{val:{fmt}}', ha='center', va='center',
                        fontsize=8, color=color)
            else:
                ax.text(j, i, '—', ha='center', va='center',
                        fontsize=9, color='gray')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

In [ ]:
# --- Performance heatmaps (episode score + Crafter score) ---
# Build grids from score_data
score_grid = make_grid(p1_vals, p2_vals, score_data,
                       lambda v: v['mean_score'])
crafter_grid = make_grid(p1_vals, p2_vals, score_data,
                         lambda v: v['crafter_score'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Training Performance (smoothed w={SMOOTH_WINDOW}) [{filter_desc}]',
             fontsize=14, fontweight='bold', y=1.02)

plot_heatmap(axes[0], score_grid, 'Episode Score (smoothed, final)', 'RdYlGn', fmt='.2f')
plot_heatmap(axes[1], crafter_grid, 'Crafter Score %', 'copper_r', fmt='.2f')

fig.tight_layout()
plt.show()

In [7]:
# --- Build spatial metric grids ---

# Decoder Manhattan error (mean)
manhattan_grid = make_grid(p1_vals, p2_vals, decode_data,
                           lambda v: np.mean(v))

# Mean SI
si_grid = make_grid(p1_vals, p2_vals, tuning_data,
                    lambda v: np.nanmean(v['metrics']['SI']))

# Mean EV
ev_grid = make_grid(p1_vals, p2_vals, tuning_data,
                    lambda v: np.nanmean(v['metrics']['EVs']))

# Mean SI for top-EV neurons (80th percentile)
def si_top_ev(layer_data):
    si = layer_data['metrics']['SI']
    ev = layer_data['metrics']['EVs']
    valid = np.isfinite(si) & np.isfinite(ev)
    if valid.sum() == 0:
        return None
    threshold = np.percentile(ev[valid], EV_PERCENTILE)
    mask = valid & (ev >= threshold)
    return np.nanmean(si[mask]) if mask.sum() > 0 else None

si_top_ev_grid = make_grid(p1_vals, p2_vals, tuning_data, si_top_ev)

# Cell type fractions
CELL_TYPES = ['untuned', 'HD_cells', 'single_field', 'border_cells',
              'spatial_HD', 'complex_cells', 'dead']

celltype_grids = {}
for ct in CELL_TYPES:
    def extract_frac(layer_data, _ct=ct):
        mask = layer_data['cell_groups'][_ct]
        return mask.sum() / len(mask) if len(mask) > 0 else None
    celltype_grids[ct] = make_grid(p1_vals, p2_vals, tuning_data, extract_frac)

print("Metric grids built.")

Metric grids built.


In [ ]:
# --- Spatial metric heatmaps (2x2) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'Sweep Metrics — Layer: {LAYER} [{filter_desc}]', fontsize=14, fontweight='bold', y=1.01)

plot_heatmap(axes[0, 0], manhattan_grid, 'Decoder Manhattan Error', 'YlOrRd')
plot_heatmap(axes[0, 1], si_grid, 'Mean Spatial Info (SI)', 'viridis')
plot_heatmap(axes[1, 0], ev_grid, 'Mean EV Reliability', 'cividis')
plot_heatmap(axes[1, 1], si_top_ev_grid, f'Mean SI (top {EV_PERCENTILE}% EV neurons)', 'plasma')

fig.tight_layout()
plt.show()

In [ ]:
# --- Cell type fraction heatmaps ---
ct_cmaps = {
    'untuned': 'Greys',
    'HD_cells': 'Oranges',
    'single_field': 'Blues',
    'border_cells': 'Greens',
    'spatial_HD': 'Purples',
    'complex_cells': 'RdPu',
    'dead': 'bone_r',
}

# Skip types that are all zero/nan
active_types = [ct for ct in CELL_TYPES
                if np.nanmax(celltype_grids[ct]) > 0]

ncols = min(4, len(active_types))
nrows = (len(active_types) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
fig.suptitle(f'Cell Type Fractions — Layer: {LAYER} [{filter_desc}]', fontsize=14, fontweight='bold', y=1.01)

axes_flat = np.array(axes).flatten() if len(active_types) > 1 else [axes]
for idx, ct in enumerate(active_types):
    plot_heatmap(axes_flat[idx], celltype_grids[ct], ct, ct_cmaps.get(ct, 'viridis'), fmt='.2f')

# Hide unused axes
for idx in range(len(active_types), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.tight_layout()
plt.show()